In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/.gitignore
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/README.md
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/requirements.txt
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/.env.example
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/tests/test_pipeline.py
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/scripts/evaluate.py
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/scripts/ingest_docs.py
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/deploy/Dockerfile
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/deploy/azure-deploy.sh
/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa/notebooks/02_QLoRA_Finetuning.ipynb
/kaggle/input/datasets/yuvrajchulpar

In [8]:
!pip install -q pymupdf loguru tqdm rank_bm25 \
    langchain langchain-community langchain-huggingface \
    sentence-transformers faiss-cpu transformers \
    peft trl bitsandbytes accelerate datasets \
    rouge-score python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, whi

# Cell 1 — Copy files to working directory & fix the broken file

In [1]:
import shutil, os

BASE = '/kaggle/input/datasets/yuvrajchulpar/rag-document-qa-by-yuvi/rag-document-qa'
WORK = '/kaggle/working/rag-document-qa'

# Copy entire project to writable working directory
shutil.copytree(BASE, WORK)
os.chdir(WORK)
print("Working directory:", os.getcwd())

Working directory: /kaggle/working/rag-document-qa


# Cell 2 — Fix the broken __init__.py & create missing ones

In [2]:
# The brace-expanded filename needs to be deleted and proper ones created
broken = os.path.join(WORK, "src/{ingestion,embeddings,retrieval,generation,finetuning,api}/__init__.py")
if os.path.exists(broken):
    os.remove(broken)
    print("Removed broken file")

# Ensure all __init__.py files exist
modules = [
    'src',
    'src/ingestion',
    'src/retrieval', 
    'src/pipeline',
    'src/finetuning',
    'src/api',
]
for mod in modules:
    init = os.path.join(WORK, mod, '__init__.py')
    os.makedirs(os.path.dirname(init), exist_ok=True)
    if not os.path.exists(init):
        open(init, 'w').close()
        print(f"Created: {init}")
    else:
        print(f"OK: {init}")

Removed broken file
OK: /kaggle/working/rag-document-qa/src/__init__.py
OK: /kaggle/working/rag-document-qa/src/ingestion/__init__.py
OK: /kaggle/working/rag-document-qa/src/retrieval/__init__.py
OK: /kaggle/working/rag-document-qa/src/pipeline/__init__.py
OK: /kaggle/working/rag-document-qa/src/finetuning/__init__.py
OK: /kaggle/working/rag-document-qa/src/api/__init__.py


# Cell 3 — Install dependencies

In [8]:
# Step 1: Fix numpy first (must be done before anything else)
!pip install -q "numpy>=1.26.0" --upgrade

# Step 2: Install all packages with compatible versions for Python 3.12
!pip install -q \
    "langchain==0.2.16" \
    "langchain-community==0.2.16" \
    "langchain-huggingface==0.0.3" \
    "sentence-transformers==3.1.1" \
    "faiss-cpu" \
    "rank_bm25==0.2.2" \
    "pymupdf==1.24.10" \
    "transformers==4.44.2" \
    "peft==0.12.0" \
    "trl==0.10.1" \
    "bitsandbytes==0.43.3" \
    "accelerate==0.34.2" \
    "datasets==2.21.0" \
    "rouge-score==0.1.2" \
    "loguru==0.7.2" \
    "python-dotenv==1.0.1" \
    "scipy>=1.11.0"

print("✅ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.1 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
langchain 0.2.16 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.4.3 which is incompatible.
langchain-community 0.2.16 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.4.3 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamin

In [4]:
# Verify numpy version is correct BEFORE importing anything else
import numpy as np
print(f"numpy version: {np.__version__}")
assert tuple(int(x) for x in np.__version__.split(".")[:2]) >= (1, 26), \
    "numpy too old — restart kernel and rerun from Cell 3"

import scipy
print(f"scipy version: {scipy.__version__}")
print("✅ numpy + scipy OK")

numpy version: 2.0.2
scipy version: 1.16.3
✅ numpy + scipy OK


# Cell 4 — Verify imports work

In [21]:
rag_chain_fixed = '''
import os
from typing import Optional, List
from pathlib import Path

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_huggingface import HuggingFacePipeline
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig,
)
from peft import PeftModel
import torch
from loguru import logger

from .prompt_templates import RAG_PROMPT
from ..ingestion.pdf_loader import PDFLoader
from ..ingestion.chunker import ResearchPaperChunker
from ..retrieval.embeddings import get_embedding_model
from ..retrieval.vector_store import FAISSVectorStore
from ..retrieval.bm25_retriever import BM25Retriever
from ..retrieval.hybrid_retriever import HybridRetriever


class RAGPipeline:
    def __init__(
        self,
        embedding_model_name: str = None,
        llm_model_name: str = None,
        adapter_path: Optional[str] = None,
        index_path: str = "indexes/faiss",
        device: str = "auto",
        top_k: int = 5,
    ):
        self.embedding_model_name = embedding_model_name or os.getenv(
            "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
        )
        self.llm_model_name = llm_model_name or os.getenv(
            "BASE_LLM", "mistralai/Mistral-7B-Instruct-v0.2"
        )
        self.adapter_path = adapter_path or os.getenv("FINETUNED_ADAPTER_PATH")
        self.index_path = index_path
        self.device = device
        self.top_k = top_k
        self._embeddings = None
        self._faiss_store = None
        self._bm25 = None
        self._hybrid_retriever = None
        self._llm_pipeline = None
        self._chunks: List[Document] = []

    def build_index(self, pdf_dir="data/papers", save_index=True, force_rebuild=False):
        self._embeddings = get_embedding_model(self.embedding_model_name)
        self._faiss_store = FAISSVectorStore(self._embeddings)

        if not force_rebuild and Path(self.index_path).exists():
            logger.info("Loading existing FAISS index from disk...")
            self._faiss_store.load(self.index_path)
            self._chunks = list(self._faiss_store._index.docstore._dict.values())
        else:
            logger.info("Building new index from PDFs...")
            loader = PDFLoader(source_dir=pdf_dir)
            pages = loader.load_all()
            chunker = ResearchPaperChunker(chunk_size=512, chunk_overlap=64)
            self._chunks = chunker.chunk(pages)
            self._faiss_store.build(self._chunks)
            if save_index:
                self._faiss_store.save(self.index_path)

        self._bm25 = BM25Retriever.from_documents(self._chunks, k=self.top_k)
        self._hybrid_retriever = HybridRetriever(
            faiss_store=self._faiss_store,
            bm25_retriever=self._bm25,
            k=self.top_k,
        )
        self._llm_pipeline = self._load_llm()
        logger.info("RAG pipeline ready!")

    def query(self, question: str) -> dict:
        if self._llm_pipeline is None:
            raise RuntimeError("Call build_index() first.")
        logger.info(f"Query: {question}")

        # Retrieve relevant docs
        source_docs = self._hybrid_retriever.get_relevant_documents(question)

        # Build context string from retrieved chunks
        context = "\\n\\n".join([doc.page_content for doc in source_docs])

        # Format prompt manually — no LangChain chain needed
        prompt_text = RAG_PROMPT.format(context=context, question=question)

        # Run through HuggingFace pipeline directly
        response = self._llm_pipeline.pipeline(
            prompt_text,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.15,
        )
        answer = response[0]["generated_text"][len(prompt_text):].strip()

        return {
            "result": answer,
            "source_documents": source_docs,
        }

    def get_context(self, question: str) -> List[Document]:
        return self._hybrid_retriever.get_relevant_documents(question)

    def _load_llm(self) -> HuggingFacePipeline:
        logger.info(f"Loading LLM: {self.llm_model_name}")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.llm_model_name)
        tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            self.llm_model_name,
            quantization_config=bnb_config,
            device_map=self.device,
            trust_remote_code=True,
        )
        if self.adapter_path and Path(self.adapter_path).exists():
            logger.info(f"Loading QLoRA adapter from {self.adapter_path}")
            model = PeftModel.from_pretrained(model, self.adapter_path)
            model = model.merge_and_unload()
        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.15,
        )
        return HuggingFacePipeline(pipeline=pipe)
'''

with open('/kaggle/working/rag-document-qa/src/pipeline/rag_chain.py', 'w') as f:
    f.write(rag_chain_fixed)
print("✅ rag_chain.py rewritten — no LangChain chains dependency")

✅ rag_chain.py rewritten — no LangChain chains dependency


In [22]:
import sys
sys.path.insert(0, '/kaggle/working/rag-document-qa')

from src.ingestion.pdf_loader import PDFLoader
from src.ingestion.chunker import ResearchPaperChunker
from src.retrieval.embeddings import get_embedding_model
from src.retrieval.vector_store import FAISSVectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import HybridRetriever
from src.pipeline.rag_chain import RAGPipeline

print("✅ All imports successful!")

✅ All imports successful!


# Cell 5 — HuggingFace login:

In [ ]:
import os
from huggingface_hub import login

os.environ['HUGGINGFACE_TOKEN'] = 'hf_add_Your_Token'  # paste your token
login(token=os.environ['HUGGINGFACE_TOKEN'])
print("✅ Logged in")

✅ Logged in


# Cell 6 — Download a research paper:

In [37]:
import urllib.request

os.makedirs('data/papers', exist_ok=True)

# arxiv_ids = ['1706.03762']
arxiv_ids = ['2512.20363']

for arxiv_id in arxiv_ids:
    url = f'https://arxiv.org/pdf/{arxiv_id}.pdf'
    out = f'data/papers/{arxiv_id}.pdf'
    urllib.request.urlretrieve(url, out)
    print(f'✅ Downloaded: {arxiv_id}.pdf')

✅ Downloaded: 2512.20363.pdf


# Cell 7 — Build index (no LLM yet — just test retrieval first):

In [38]:
from src.ingestion.pdf_loader import PDFLoader
from src.ingestion.chunker import ResearchPaperChunker
from src.retrieval.embeddings import get_embedding_model
from src.retrieval.vector_store import FAISSVectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import HybridRetriever

# Load & chunk
loader = PDFLoader(source_dir='data/papers')
pages = loader.load_all()
print(f'Pages: {len(pages)}')

chunker = ResearchPaperChunker(chunk_size=512, chunk_overlap=64)
chunks = chunker.chunk(pages)
print(f'Chunks: {len(chunks)}')

# Build FAISS
embedding_model = get_embedding_model(device='cuda')
faiss_store = FAISSVectorStore(embedding_model)
faiss_store.build(chunks)
faiss_store.save('indexes/faiss')

# Build BM25 + Hybrid
bm25 = BM25Retriever.from_documents(chunks, k=5)
hybrid = HybridRetriever(faiss_store=faiss_store, bm25_retriever=bm25, k=5)

# Test retrieval — NO LLM needed for this step
docs = hybrid.get_relevant_documents("What is the attention mechanism?")
print(f'\n✅ Retrieved {len(docs)} docs')
for d in docs:
    print(f"  [{d.metadata['source']} p.{d.metadata['page']}] {d.page_content[:80]}...")

2026-03-21 12:04:43.798 | INFO     | src.ingestion.pdf_loader:_load_from_directory:69 - Found 2 PDFs in data/papers

Loading PDFs: 100%|██████████| 2/2 [00:00<00:00,  6.07it/s]
2026-03-21 12:04:44.132 | INFO     | src.ingestion.pdf_loader:_load_from_directory:73 - Extracted 29 pages total
2026-03-21 12:04:44.138 | INFO     | src.ingestion.chunker:chunk:64 - Chunked 29 pages → 250 chunks (size=512, overlap=64)
2026-03-21 12:04:44.139 | INFO     | src.retrieval.embeddings:get_embedding_model:35 - Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 on cuda


Pages: 29
Chunks: 250


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-21 12:04:45.986 | INFO     | src.retrieval.embeddings:get_embedding_model:46 - Embedding model loaded successfully
2026-03-21 12:04:45.988 | INFO     | src.retrieval.vector_store:build:41 - Building FAISS index from 250 chunks...
2026-03-21 12:04:46.678 | INFO     | src.retrieval.vector_store:build:52 - FAISS index built successfully
2026-03-21 12:04:46.681 | INFO     | src.retrieval.vector_store:save:67 - FAISS index saved to indexes/faiss
2026-03-21 12:04:46.695 | INFO     | src.retrieval.bm25_retriever:from_documents:23 - BM25 index built on 250 documents


AttributeError: 'HybridRetriever' object has no attribute 'get_relevant_documents'

# Cell 8 — Patch the source files first:

In [39]:
for fpath in [
    '/kaggle/working/rag-document-qa/src/retrieval/hybrid_retriever.py',
    '/kaggle/working/rag-document-qa/src/retrieval/bm25_retriever.py',
    '/kaggle/working/rag-document-qa/src/pipeline/rag_chain.py',
]:
    with open(fpath, 'r') as f:
        content = f.read()
    content = content.replace('get_relevant_documents', 'invoke')
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"✅ Patched {fpath.split('/')[-1]}")

✅ Patched hybrid_retriever.py
✅ Patched bm25_retriever.py
✅ Patched rag_chain.py


# Cell 9 — Test retrieval:

The patch renamed the internal method too. Let me rewrite bm25_retriever.py cleanly:

In [40]:
bm25_fixed = '''
from typing import List
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from rank_bm25 import BM25Okapi
from loguru import logger
import re


class BM25Retriever(BaseRetriever):
    documents: List[Document]
    bm25: object
    k: int = 5

    class Config:
        arbitrary_types_allowed = True

    @classmethod
    def from_documents(cls, documents: List[Document], k: int = 5) -> "BM25Retriever":
        tokenized_corpus = [cls._tokenize(doc.page_content) for doc in documents]
        bm25 = BM25Okapi(tokenized_corpus)
        logger.info(f"BM25 index built on {len(documents)} documents")
        return cls(documents=documents, bm25=bm25, k=k)

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_k_indices = sorted(
            range(len(scores)), key=lambda i: scores[i], reverse=True
        )[:self.k]
        results = []
        for idx in top_k_indices:
            doc = self.documents[idx]
            doc.metadata["bm25_score"] = float(scores[idx])
            results.append(doc)
        return results

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.findall(r"[a-z0-9]+", text.lower())
'''

with open('/kaggle/working/rag-document-qa/src/retrieval/bm25_retriever.py', 'w') as f:
    f.write(bm25_fixed)
print("✅ bm25_retriever.py rewritten")

✅ bm25_retriever.py rewritten


Then rewrite hybrid_retriever.py cleanly too:

In [41]:
hybrid_fixed = '''
from typing import List, Dict
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks.manager import CallbackManagerForRetrieverRun
from loguru import logger

from .vector_store import FAISSVectorStore
from .bm25_retriever import BM25Retriever


class HybridRetriever(BaseRetriever):
    faiss_store: FAISSVectorStore
    bm25_retriever: BM25Retriever
    k: int = 5
    rrf_k: int = 60
    dense_weight: float = 0.6
    sparse_weight: float = 0.4

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> List[Document]:
        fetch_k = self.k * 3
        dense_docs  = self.faiss_store.similarity_search(query, k=fetch_k)
        sparse_docs = self.bm25_retriever._get_relevant_documents(query)
        fused = self._rrf(dense_docs, sparse_docs)
        logger.debug(f"Hybrid: dense={len(dense_docs)}, sparse={len(sparse_docs)}, returning top {self.k}")
        return fused[:self.k]

    def _rrf(self, dense_docs: List[Document], sparse_docs: List[Document]) -> List[Document]:
        scores: Dict[str, float] = {}
        doc_map: Dict[str, Document] = {}
        for rank, doc in enumerate(dense_docs):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + self.dense_weight * (1.0 / (self.rrf_k + rank + 1))
            doc_map[key] = doc
        for rank, doc in enumerate(sparse_docs):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + self.sparse_weight * (1.0 / (self.rrf_k + rank + 1))
            doc_map[key] = doc
        sorted_keys = sorted(scores, key=scores.__getitem__, reverse=True)
        results = []
        for key in sorted_keys:
            doc = doc_map[key]
            doc.metadata["rrf_score"] = round(scores[key], 6)
            results.append(doc)
        return results
'''

with open('/kaggle/working/rag-document-qa/src/retrieval/hybrid_retriever.py', 'w') as f:
    f.write(hybrid_fixed)
print("✅ hybrid_retriever.py rewritten")

✅ hybrid_retriever.py rewritten


Then re-import and rerun:

In [43]:
# Reload modules
import importlib
import src.retrieval.bm25_retriever as bm25_mod
import src.retrieval.hybrid_retriever as hybrid_mod
importlib.reload(bm25_mod)
importlib.reload(hybrid_mod)

from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import HybridRetriever

# Rebuild with fixed classes
bm25   = BM25Retriever.from_documents(chunks, k=5)
hybrid = HybridRetriever(faiss_store=faiss_store, bm25_retriever=bm25, k=5)

# Test
docs = hybrid.invoke("What is the cluster federated Learning?")
print(f'✅ Retrieved {len(docs)} docs')
for d in docs:
    print(f"  [{d.metadata['source']} p.{d.metadata['page']}] {d.page_content[:80]}...")

2026-03-21 12:05:53.452 | INFO     | src.retrieval.bm25_retriever:from_documents:23 - BM25 index built on 250 documents
2026-03-21 12:05:53.464 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5


✅ Retrieved 5 docs
  [2512.20363.pdf p.12] [21] J. Wolfrath, N. Sreekumar, D. Kumar, Y. Wang, and A. Chandra,
“HACCS: Heter...
  [2512.20363.pdf p.3] dom selection, it achieves faster convergence, higher accuracy,
and more stable ...
  [2512.20363.pdf p.3] to minimize cross-cluster similarity and repeats; the method
is model-agnostic, ...
  [1706.03762.pdf p.15] The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
wha...
  [2512.20363.pdf p.12] IEEE, 2022, pp.
1–5.
[23] F. Sattler, K.-R. M¨uller, and W. Samek, “Clustered fe...


# Cell 10 — Load Mistral + Query:

In [33]:
from src.pipeline.rag_chain import RAGPipeline

rag = RAGPipeline(
    index_path='indexes/faiss',
    top_k=5,
)

# Skip re-building index — just reload it from disk
rag._embeddings        = embedding_model
rag._faiss_store       = faiss_store
rag._chunks            = chunks
rag._bm25              = bm25
rag._hybrid_retriever  = hybrid

# Load LLM only
print("Loading Mistral-7B in 4-bit... (~5 mins on T4)")
rag._llm_pipeline = rag._load_llm()
print("✅ Mistral-7B loaded!")

2026-03-21 11:46:52.855 | INFO     | src.pipeline.rag_chain:_load_llm:115 - Loading LLM: mistralai/Mistral-7B-Instruct-v0.2


Loading Mistral-7B in 4-bit... (~5 mins on T4)


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Mistral-7B loaded!


# Cell 11 — Ask questions:

In [44]:
from src.pipeline.prompt_templates import RAG_PROMPT

def ask(question, hybrid_retriever, llm_pipeline):
    # Step 1: Retrieve
    source_docs = hybrid_retriever._get_relevant_documents(question)
    
    # Step 2: Build context
    context = "\n\n".join([doc.page_content for doc in source_docs])
    
    # Step 3: Format prompt
    prompt_text = RAG_PROMPT.format(context=context, question=question)
    
    # Step 4: Run LLM
    response = llm_pipeline.pipeline(
        prompt_text,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        repetition_penalty=1.15,
    )
    answer = response[0]["generated_text"][len(prompt_text):].strip()
    
    return {"result": answer, "source_documents": source_docs}


# Ask questions directly
questions = [
    "What is the cluster federated Learning?",
    "What datasets were used for evaluation?",
    "What are the main results of this paper?",
]

for q in questions:
    result = ask(q, hybrid, rag._llm_pipeline)
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print(f"Sources: {[d.metadata['source'] + ' p.' + str(d.metadata['page']) for d in result['source_documents']]}")
    print("-" * 60)

2026-03-21 12:07:00.750 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-03-21 12:07:11.314 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the cluster federated Learning?
A: Cluster Federated Learning (CFL) is a method for handling non-IID data in federated learning. It recursively clusters clients via the cosine similarity of their local updates when training stalls, aiming to minimize cross-cluster similarity (Sattler et al., 2020). This approach is model-agnostic, FedAvg-compatible, and does not require a preset number of clusters.
Sources: ['2512.20363.pdf p.12', '2512.20363.pdf p.3', '2512.20363.pdf p.3', '1706.03762.pdf p.15', '2512.20363.pdf p.12']
------------------------------------------------------------


2026-03-21 12:07:17.998 | DEBUG    | src.retrieval.hybrid_retriever:_get_relevant_documents:30 - Hybrid: dense=15, sparse=5, returning top 5
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What datasets were used for evaluation?
A: The evaluation was conducted on six datasets: ACSIncome, Serengeti, FMNIST, CIFAR10, Sent140, and Amazon reviews (Refer Table I on pages 23-24 for detailed information).
Sources: ['2512.20363.pdf p.13', '2512.20363.pdf p.7', '2512.20363.pdf p.6', '2512.20363.pdf p.13', '2512.20363.pdf p.6']
------------------------------------------------------------

Q: What are the main results of this paper?
A: The authors propose a method called Attention Is All You Need and compare it to various baselines in terms of global accuracy and fairness metrics (AD, SDAD) across different datasets and non-IID settings. They report that their proposed method consistently outperforms the baselines, achieving up to 18% relative gains over strong contenders such as CFL and FedSoft (Vaswani et al., 2017).
Sources: ['1706.03762.pdf p.15', '1706.03762.pdf p.1', '2512.20363.pdf p.14', '2512.20363.pdf p.4', '2512.20363.pdf p.8']
---------------------------------------